In [1]:
"""
================================================================================
Log Template Extraction Module (AIOps-style, Dataset-agnostic)
================================================================================
1. Recursively scans a dataset directory to discover all container log files
   named:
       `log_filebeat-testbed-log-service.csv`

2. Excludes test data (e.g., paths containing "test_data") to ensure that
   **log template vocabularies are learned only from training data**.

3. For each log entry:
   - Validates the log record (entity filtering, text validity)
   - Maps pod-level `cmdb_id` to service-level identifier
   - Determines the programming language of the service
   - Uses a **language-specific Drain3 TemplateMiner** to extract a log template

4. Maintains a **separate template vocabulary for each programming language**,
   preventing template pollution across heterogeneous log formats.

5. Outputs:
   - A JSON file containing the list of discovered log templates per language
   - A pickle file containing trained Drain3 TemplateMiner objects


------------------------
Inputs
------------------------
- data_base: Root directory of the dataset
- log_filebeat-testbed-log-service.csv: Container log files
- service_language_dict: Mapping from service name to programming language
- all_entity_list: List of valid system entities (pods/services)
- rename_pod2service: Function mapping pod name to service name
- Drain3 configuration file (.ini)

------------------------
Outputs
------------------------
提取日志模板，输出
- log_patterns.json
    {
        "Java":    [template_1, template_2, ...],
        "Go":      [...],
        "Python":  [...],
        ...
    }

- log_template_miner.pkl
    {
        language: TemplateMiner instance
    }

These outputs serve as **fixed vocabularies** for downstream log feature
generation and modeling stages.
================================================================================
"""

'\n================================================================================\nLog Template Extraction Module (AIOps-style, Dataset-agnostic)\n================================================================================\n1. Recursively scans a dataset directory to discover all container log files\n   named:\n       `log_filebeat-testbed-log-service.csv`\n\n2. Excludes test data (e.g., paths containing "test_data") to ensure that\n   **log template vocabularies are learned only from training data**.\n\n3. For each log entry:\n   - Validates the log record (entity filtering, text validity)\n   - Maps pod-level `cmdb_id` to service-level identifier\n   - Determines the programming language of the service\n   - Uses a **language-specific Drain3 TemplateMiner** to extract a log template\n\n4. Maintains a **separate template vocabulary for each programming language**,\n   preventing template pollution across heterogeneous log formats.\n\n5. Outputs:\n   - A JSON file containing the

In [1]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import os
import pandas as pd
import json
import pickle

In [2]:
def extract_log_patterns(
    data_base: str,
    drain_ini_path: str,
    all_entity_list: list,
    service_language_dict: dict,
    rename_pod2service,
    out_patterns_json: str,
    out_miner_pkl: str,
    skip_if_path_contains=("test_data",),
    chunksize: int = 100_000,
):
    # 1) 初始化 Drain3 配置
    drain_config = TemplateMinerConfig()
    drain_config.load(drain_ini_path)
    drain_config.profiling_enabled = True

    # 2) 自动得到语言集合（适配不同数据集）
    languages = sorted(set(service_language_dict.values()))

    # 3) 为每种语言创建 miner + pattern 容器
    miners = {lang: TemplateMiner(config=drain_config) for lang in languages}

    # 用 set 去重（快），再配合 list 保留顺序
    pattern_sets = {lang: set() for lang in languages}
    pattern_lists = {lang: [] for lang in languages}

    target_name = "log_filebeat-testbed-log-service.csv"

    # 4) 扫目录找到目标文件
    for root, _, files in os.walk(data_base):
        if target_name not in files:
            continue

        log_path = os.path.join(root, target_name)

        # 过滤 test_data
        if any(x in log_path for x in skip_if_path_contains):
            continue

        # 5) 分块读取
        reader = pd.read_csv(log_path, chunksize=chunksize)

        for chunk in reader:
            # 用 itertuples 更快
            for row in chunk.itertuples(index=False):
                # 假设列名包含 cmdb_id / value
                cmdb_id = getattr(row, "cmdb_id", None)
                value = getattr(row, "value", None)

                if cmdb_id is None or value is None:
                    continue
                if cmdb_id not in all_entity_list:
                    continue
                if not isinstance(value, str):
                    continue

                service = rename_pod2service(cmdb_id)
                lang = service_language_dict.get(service)
                if lang is None:
                    continue

                template = miners[lang].add_log_message(value).get("template_mined", "")
                if not template:
                    continue

                # 6) 去重并保序
                if template not in pattern_sets[lang]:
                    pattern_sets[lang].add(template)
                    pattern_lists[lang].append(template)

    # 7) 保存 patterns（json）和 miners（pkl）
    os.makedirs(os.path.dirname(out_patterns_json), exist_ok=True)
    with open(out_patterns_json, "w", encoding="utf-8") as f:
        json.dump(pattern_lists, f, ensure_ascii=False, indent=2)

    os.makedirs(os.path.dirname(out_miner_pkl), exist_ok=True)
    with open(out_miner_pkl, "wb") as f:
        pickle.dump(miners, f)

    print(f"[OK] extracted patterns -> {out_patterns_json}")
    print(f"[OK] saved miners -> {out_miner_pkl}")

    return pattern_lists, miners

In [3]:
all_entity_list = [
    'node-1', 
    'node-2', 
    'node-3', 
    'node-4', 
    'node-5', 
    'node-6', 
    'adservice', 
    'cartservice', 
    'checkoutservice', 
    'currencyservice', 
    'emailservice', 
    'frontend', 
    'paymentservice', 
    'productcatalogservice', 
    'recommendationservice', 
    'shippingservice', 
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [4]:
service_language_dict = {
    'adservice': 'Java', 
    'cartservice': 'C#', 
    'checkoutservice': 'Go', 
    'currencyservice': 'Node.js', 
    'emailservice': 'Python', 
    'frontend': 'Go', 
    'paymentservice': 'Node.js', 
    'productcatalogservice': 'Go', 
    'recommendationservice': 'Python', 
    'shippingservice': 'Go'}

In [5]:
data_base = "/home/ZZData/Eadro/preprocess/datasets/aiops2022/"

# 去除pod后缀变成service
def rename_pod2service(pod_name):
    return pod_name.replace('2-0', '') \
                   .replace('-0', '') \
                   .replace('-1', '') \
                   .replace('-2', '')

patterns, miners = extract_log_patterns(
    data_base=data_base,
    drain_ini_path="./aiops22_drain3.ini",
    all_entity_list=all_entity_list,
    service_language_dict=service_language_dict,
    rename_pod2service=rename_pod2service,
    out_patterns_json="../../temp_data/aiops22/analysis/log/log_patterns.json",
    out_miner_pkl="../../temp_data/aiops22/analysis/log/log_template_miner.pkl",
)


total          : took    23.44 s (100.00%),  2,052,591 samples,   11.42 ms / 1000 samples,       87,564.60 hz
drain          : took    10.44 s ( 44.56%),  2,052,591 samples,    5.09 ms / 1000 samples,      196,523.98 hz
mask           : took     9.11 s ( 38.85%),  2,052,591 samples,    4.44 ms / 1000 samples,      225,386.08 hz
tree_search    : took     4.66 s ( 19.87%),  2,052,591 samples,    2.27 ms / 1000 samples,      440,684.94 hz
cluster_exist  : took     2.27 s (  9.69%),  2,052,569 samples,    1.11 ms / 1000 samples,      903,642.84 hz
create_cluster : took     0.00 s (  0.00%),         22 samples,    7.88 ms / 1000 samples,      126,925.29 hz
total          : took    12.16 s (100.00%),  1,272,304 samples,    9.56 ms / 1000 samples,      104,608.90 hz
drain          : took     5.41 s ( 44.51%),  1,272,304 samples,    4.25 ms / 1000 samples,      235,041.89 hz
mask           : took     4.31 s ( 35.47%),  1,272,304 samples,    3.39 ms / 1000 samples,      294,882.03 hz
tree_searc